In [ ]:
pip install stargazer openpyxl

In [ ]:
import os 
current_dir = os.getcwd()
print("Current Directory:", current_dir)

In [ ]:
new_dir = os.path.abspath(os.path.join(current_dir, "../../.."))
os.chdir(new_dir)
print("New Directory:", os.getcwd())

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from scipy.stats import ttest_ind
from scipy import stats
import statsmodels.formula.api as smf
import statsmodels.api as sm
import matplotlib.pyplot as plt

from stargazer.stargazer import Stargazer, LineLocation

from img_labormarket.config import BLD, SRC
from img_labormarket.analysis.help_analysis_new import *
from img_labormarket.data_management.help_data import *
# Setze die Anzeigeoptionen für Pandas
pd.set_option('display.float_format', '{:.3f}'.format)

green = '#000080'
red = '#ff69b4'

In [ ]:
df = pd.read_csv(BLD / 'data' / 'clean_data.csv')

# Job Search 

For this analysis, we focus on our sample of Master students.

In [ ]:
df = df[~df['Gender'].isin(['Prefer not to say', 'Divers'])]
df['Gender'].value_counts()

In [ ]:
df.loc[df['StudyFieldCluster']== 'Natural & Life Sciences'].groupby(['Gender','sector_coarse'])[['first_wage']].value_counts().unstack(0)

In [ ]:
df = df.loc[df['DegreeCompact']=='Master']
df['Gender'].value_counts()

In [ ]:
df['StudyFieldCluster'].value_counts()

In [ ]:
df.groupby('Gender')['StudyFieldCluster'].value_counts()

# Facts

In [ ]:
# Split the data
df_female = df.loc[df['Gender'] == 'Female', 'DiffGradJobStart'].sort_values()
df_male = df.loc[df['Gender'] == 'Male', 'DiffGradJobStart'].sort_values()
# Create the CDFs
cdf_female = pd.Series(dtype=float)
cdf_male = pd.Series(dtype=float)

for month in range(-5, df['DiffGradJobStart'].max() + 1):
    cdf_female.loc[month] = (df_female <= month).mean()
    cdf_male.loc[month] = (df_male <= month).mean()


# Perform the Kolmogorov-Smirnov test
ks_stat, p_value = stats.ks_2samp(df_female, df_male)
p_value = "<0.001" if p_value < 0.001 else f"{p_value:.3f}"
# Replot with increased font size and KS test results annotated
plt.figure(figsize=(16, 9))
plt.axvline(0, color='grey', linestyle='--', linewidth=3)
plt.plot(cdf_female.loc[:20], marker='o',color=red ,label='Females',linewidth=4.5, markersize=10)
plt.plot(cdf_male.loc[:20], marker='s',color=green ,label='Males',linewidth=4.5, markersize=10)

# Increase font size
plt.xlabel("Months relative to graduation", fontsize=22)
plt.ylabel("Proportion Started Job", fontsize=22)
plt.xticks(fontsize=22)
plt.yticks(fontsize=22)
plt.ylim(0, 1)
plt.legend(fontsize=22, loc='lower right')
plt.grid()
# Annotate KS test results
plt.text(14.5, 0.25, f"KS Statistic: {ks_stat:.3f}\nP-value: {p_value}", 
         fontsize=22, bbox=dict(facecolor='white', alpha=0.8, edgecolor='grey', boxstyle="round,pad=0.1"))

plt.savefig(BLD / 'figures' / 'cdf_grad_job_start.pdf', bbox_inches='tight')
# Show plot
plt.show()

In [ ]:
df = pd.read_csv(BLD / 'data' / 'clean_data.csv')
df = df.loc[~df['Gender'].isin(['Prefer not to say'])]
df = df.loc[df['DegreeCompact']=='Master']
res0 = smf.ols("DiffGradJobStart ~ img", data=df).fit(cov_type='HC1')
res1 = smf.ols("DiffGradJobStart ~ img + C(Gender, Treatment('Male')) + C(StudyFieldCluster) + C(Age)", data=df).fit(cov_type='HC1')
res2 = smf.ols("DiffGradJobStart ~ img + C(Gender, Treatment('Male')) + C(StudyFieldCluster)+ C(Age) + C(GradYear)", data=df).fit(cov_type='HC1')
res3 = smf.ols("DiffGradJobStart ~ img + C(Gender, Treatment('Male')) + C(StudyFieldCluster) + C(sector_coarse)+ C(Age)", data=df).fit(cov_type='HC1')
res4 = smf.ols("DiffGradJobStart ~ img + C(Gender, Treatment('Male')) + C(StudyFieldCluster) + C(sector_coarse)+ C(Age) + C(GradYear)", data=df).fit(cov_type='HC1')


# Prepare the models
models = [res0, res1, res2, res3, res4]
# Define restriction labels
extras = [
    "Basic Controls",
    "Year FE",
    "Industry FE"
]

# Define whether restrictions are applied (x: yes, blank: no)
fixed_effects = [
    ["","", ""],  # res2:
    ["x","", ""],  # res4: 
    ["x","x", ""],  # res3: 
    ["x","", "x"],  # res5: 
    ["x","x", "x"]  # res6: 
]

# Create Stargazer table
stargazer = Stargazer(models)
#stargazer.title("OLS Regression Results with Restrictions")
#stargazer.custom_columns(["Model 0", "Model 1", "Model 2", "Model 3", "Model 4"], [1, 1, 1, 1, 1])

# Add restrictions as additional rows
for idx, restriction in enumerate(extras):
    row_values = [fixed_effects[model_idx][idx] for model_idx in range(len(models))]
    stargazer.add_line(restriction, row_values)

mean = df['DiffGradJobStart'].mean().round(3)
stargazer.rename_covariates({
    "img":"Immigrant",
})
stargazer.covariate_order(["img",])

stargazer.show_residual_std_err = False
stargazer.show_f_statistic = False
stargazer.dependent_variable_name = "Month of Job Acceptance (relative to graduation)"
stargazer.add_line('Mean', [mean]*5, LineLocation.FOOTER_TOP)
code = stargazer.render_latex()
#with open(BLD / 'tables' / 'job_search_table.tex', 'w') as f:
#    f.write(code)
stargazer


# Characteristics of the first job
    ## English? # Proportions Test
    ## Size of the firm? Mannn-Whitney U Test
    ## Degree as a requirement # Proportions Test ???
    ## ???

# Job Search
    4 ## Number of Applications
        #1. More than 50 # Proportions test x
        #2. Less than 5 # Proportions test x
    5 ## Number of internships # Count 5 or more as 5 show point estimate x
    6 ## Search Radius # MW-Test x
   1 ## Job Search Duration  x
    #NAH ## Start Job Search before or after graduation # Proportions Test 
   3 ## Job Acceptance # Proportions Test x
   2 ## Reservation Wage # t-Test x

In [ ]:
rslt1 = male_female_comparison(df, ['job_search_months','real_first_wage'])
rslt1.rename(columns={'Male Avg': 'Male', 'Female Avg': 'Female'},inplace=True)
rslt1.rename(index={'job_search_months':'Job Search (in months)'},inplace=True)
rslt1 = rslt1[['Male','Female','p-value']]
rslt1

In [ ]:
df_w = restrict_wage(df, 'real_reservation_wage', 15, 150)
rslt2 = male_female_comparison(df_w, ['real_reservation_wage'])
rslt2.rename(columns={'Male Avg': 'Male', 'Female Avg': 'Female'},inplace=True)
rslt2.rename(index={'real_reservation_wage':'Initial Reservation Wage'},inplace=True)
rslt2 = rslt2[['Male','Female','p-value']]
rslt2

In [ ]:
def compare_offer_acceptance(df):
    rslt = pd.DataFrame({'Male':[],'Female':[],'p-value':[]})
    df_female = df.loc[df['Gender'] == 'Female','Q23'].dropna()
    df_male = df.loc[df['Gender'] == 'Male','Q23'].dropna()

    count = np.array([df_male.value_counts().loc[1],df_female.value_counts().loc[1]])
    nobs = np.array([len(df_male),len(df_female)])
    z_stat, p_value = sm.stats.proportions_ztest(count, nobs,alternative='two-sided')

    fraction = (count / nobs)*100
    p_value = '$<$0.001' if float(p_value) < 0.001 else f'{p_value:.3f}'
    new_row1 = pd.DataFrame({'Male': f'{fraction[0]:.2f}', 'Female': f'{fraction[1]:.2f}', 'p-value': p_value},index=['Accept first offer'])
    rslt = pd.concat([rslt,new_row1])
    return rslt

rslt3 = compare_offer_acceptance(df)
rslt3

In [ ]:
def compare_applications(df):
    # Split data
    df_female = df.loc[df['Gender'] == 'Female','no_applications'].dropna()
    df_male = df.loc[df['Gender'] == 'Male', 'no_applications'].dropna()
    
    # Observations 
    nobs = np.array([
        len(df_male),
        len(df_female)])
    count_more_than_50 = np.array([
        df_male.value_counts().loc['More than 50 applications'],
        df_female.value_counts().loc['More than 50 applications'],
    ])
    'Less than 5 applications','Between 5 and 10 applications'
    count_less_than_10 = np.array([
        (df_male.value_counts().loc['Less than 5 applications']+df_male.value_counts().loc['Between 5 and 10 applications']),
        (df_female.value_counts().loc['Less than 5 applications']+df_female.value_counts().loc['Between 5 and 10 applications'])
    ])
    rslt = pd.DataFrame({'Male':[],'Female':[],'p-value':[]})

    fraction_more_than_50 = (count_more_than_50 / nobs)*100
    z_stat, p_value_more_than_50 = sm.stats.proportions_ztest(count_more_than_50, nobs, alternative='two-sided')
    p_value_more_than_50 = '$<$0.001' if float(p_value_more_than_50) < 0.001 else f'{p_value_more_than_50:.3f}'
    new_row1 = pd.DataFrame({'Male': f'{fraction_more_than_50[0]:.2f}', 'Female': f'{fraction_more_than_50[1]:.2f}', 'p-value': p_value_more_than_50},index=['More than 50 applications'])
    fraction_less_than_10 = (count_less_than_10 / nobs)*100
    z_stat, p_value_less_than_10 = sm.stats.proportions_ztest(count_less_than_10, nobs, alternative='two-sided')
    p_value_less_than_10 = '$<$0.001' if float(p_value_less_than_10) < 0.001 else f'{p_value_less_than_10:.3f}'
    new_row2 = pd.DataFrame({'Male': f'{fraction_less_than_10[0]:.2f}', 'Female': f'{fraction_less_than_10[1]:.2f}', 'p-value': p_value_less_than_10},index=['Less than 10 applications'])
    rslt = pd.concat([rslt,new_row2])
    rslt = pd.concat([rslt,new_row1])

    return rslt

rslt4 = compare_applications(df)
rslt4

In [ ]:
mapping_df = pd.read_excel('data/ValueCodings.xlsx')

mapping_df['Frage'].fillna(method='ffill',inplace=True)
df = add_codings(df,mapping_df,'Q16','no_internships')
df.groupby('Gender')[['no_internships']].value_counts(normalize=True).unstack(0)

df['no_internships'] = df['no_internships'].replace({'5 Internships or more':5,
                                                    'No internship':0,
                                                    '1 internship':1,
                                                    '2 internships':2,
                                                    '3 internships':3,
                                                    '4 Internships':4})
rslt5 = male_female_comparison(df,['no_internships'])
rslt5.rename(columns={'Male Avg': 'Male', 'Female Avg': 'Female'},inplace=True)
rslt5.rename(index={'no_internships':'Number of Internships'},inplace=True) 
rslt5 = rslt5[['Male','Female','p-value']]
rslt5['p-value'] = rslt5['p-value'].apply(lambda x: '$<$0.001' if float(x) < 0.001 else f'{x:.3f}')
rslt5

In [ ]:
def compare_search_radius(df):
    df_female = df.loc[df['Gender'] == 'Female','search_distance'].dropna()
    female = df_female.replace({'< 20 km':0,
                                 '< 50 km':1,
                                 '< 100 km':2,
                                 'within Germany':3,
                                 'within the EU':4,
                                 'all over the world':5})
    df_male = df.loc[df['Gender'] == 'Male','search_distance'].dropna()
    male = df_male.replace({'< 20 km':0,
                                 '< 50 km':1,
                                 '< 100 km':2,
                                 'within Germany':3,
                                 'within the EU':4,
                                 'all over the world':5})
    
    # Observations 
    nobs = np.array([
        len(df_male),
        len(df_female)])

    rslt = pd.DataFrame({'Male':[],'Female':[],'p-value':[]})
    for radius in ['< 20 km','< 50 km','< 100 km','within Germany','within the EU','all over the world']:
        # Count obs
        count_male = df_male.value_counts().loc[radius] if radius in df_male.value_counts().index else 0
        count_female = df_female.value_counts().loc[radius] if radius in df_female.value_counts().index else 0
        count = np.array([
            count_male,
            count_female,
                       ])
        fraction = (count / nobs)*100
        p_val = np.nan
        if radius == '< 20 km':
            radius = 'Less than 20 km'
        elif radius == '< 50 km':
            radius = 'Less than 50 km'
        elif radius == '< 100 km':
            radius = 'Less than 100 km'
            # Perform Mann-Whitney U test
            p_val = 'MW-Test:'
        elif radius == 'within Germany':
            stat, p_val = stats.mannwhitneyu(male, female, alternative="two-sided")
            p_val = '$<$0.001' if p_val < 0.001 else f'{p_val:.3f}'
        new_row = pd.DataFrame({'Male': f'{fraction[0]:.2f}', 'Female': f'{fraction[1]:.2f}', 'p-value': p_val},index=[radius])
        rslt = pd.concat([rslt,new_row])

    return rslt

rslt6 = compare_search_radius(df)
rslt6

In [ ]:
rslt7 = male_female_comparison(df, ['Q43r1'])

rename_dict = {
    'Q43r1': 'Average Salary Estimation'
}
rslt7.rename(index=rename_dict, inplace=True)
rslt7[['Female Avg', 'Male Avg', 'p-value']]
rslt7['p-value'] = rslt7['p-value'].apply(lambda x: '$<$0.001' if float(x) < 0.001 else f'{x:.3f}')
rslt7.rename(columns={'Male Avg': 'Male', 'Female Avg':'Female'},inplace=True)
rslt7

In [ ]:
pd.isna(rslt6.loc['within the EU','p-value'])

In [ ]:
def generate_latex_table(rslt1,rslt2,rslt3,rslt4,rslt5,rslt6,rslt7, filename="js_table.tex"):
    """
    Generate a LaTeX table from a DataFrame with numerical values rounded
    to two decimal places and p-values to three decimal places.
    
    Parameters:
    df (pd.DataFrame): The data to format.
    filename (str): The output .tex file name.
    """
    # LaTeX Table Header
    latex_code = r"""
        \begin{tabular}{lccc}
            \toprule
             & Male &  Female & p-value \\
            \midrule
    """
    # Add rows dynamically
    for locs, row in rslt1.iterrows():
        latex_code += f"        \\textbf{{{locs}}} & {float(row['Male']):.2f} & {float(row['Female']):.2f} & {float(row['p-value']):.3f} \\\ \n"
    for locs, row in rslt2.iterrows():
        latex_code += f"        \\textbf{{{locs}}} & {float(row['Male']):.2f} & {float(row['Female']):.2f} & {row['p-value']} \\\ \n"
    for locs, row in rslt7.iterrows():
        latex_code += f"        \\textbf{{{locs}}} & {float(row['Male']):.2f} & {float(row['Female']):.2f} & {row['p-value']} \\\ \n"
    for locs, row in rslt3.iterrows():
        latex_code += f"        \\textbf{{{locs}}} & {row['Male']}\% & {row['Female']}\% & {row['p-value']} \\\ \n"
    latex_code += f"  \\textbf{{Applications}} &  & &  \\\ \n"
    for locs, row in rslt4.iterrows():
        latex_code += f"\\phantom{{00}}        {locs} & {row['Male']}\% & {row['Female']}\% & {row['p-value']} \\\ \n"
    for locs, row in rslt5.iterrows():
        latex_code += f"\\textbf{{{locs}}} & {float(row['Male']):.2f} & {float(row['Female']):.2f} & {float(row['p-value']):.3f}  \\\ \n"
    latex_code += f"  \\textbf{{Search Radius}} &  & &  \\\ \n"
    for locs, row in rslt6.iterrows():
        if pd.isna(row['p-value']):
            latex_code += f"\\phantom{{00}}        {locs} & {row['Male']}\% & {row['Female']}\% &  \\\ \n"
        else:
            latex_code += f"\\phantom{{00}}        {locs} & {row['Male']}\% & {row['Female']}\% & {row['p-value']} \\\ \n"
    # LaTeX Table Footer
    latex_code += r"""        \bottomrule
    \end{tabular}
    """
    
    # Save to .tex file
    with open(filename, "w") as f:
        f.write(latex_code)
    
    return filename
dir = new_dir + "\\img_labormarket\\analysis\\Gender analysis\\Charts and tables\\job_search_moments.tex"
generate_latex_table(rslt1,rslt2,rslt3,rslt4,rslt5,rslt6,rslt7, filename=dir)

# Characteristics of the first job

### How many employees?

1	Less than 20 employees

2	Between 20 to 50 employees

3	Between 50 to 200 employees

4	Between 200 to 500 employees

5	Between 500 to 1,000 employees

6	Between 1,000 to 5,000 employees

7	Between 5,000 to 10,000 employees

8	More than 10,000

In [ ]:
df = add_codings(df,mapping_df,'Q12_1c1','university')
x = df.groupby('Gender')['university'].value_counts().unstack(0)

In [ ]:
df['firmsize'] = df['Q33'].replace({1:'Less than 20 employees',
                                    2:'Between 20 to 200 employees',
                                    3:'Between 20 to 200 employees',
                                    4: 'Between 200 to 1,000 employees',
                                    5: 'Between 200 to 1,000 employees',
                                    6: 'Between 1,000 to 5,000 employees',
                                    7: 'Between 5,000 to 10,000 employees',
                                    8: 'More than 10,000 employees',})
rslt1 = df.groupby('Gender')['firmsize'].value_counts(normalize=True).unstack(0).loc[['Less than 20 employees','Between 20 to 200 employees','Between 200 to 1,000 employees','Between 1,000 to 5,000 employees','Between 5,000 to 10,000 employees','More than 10,000 employees']]

stat, p_value = stats.mannwhitneyu(df.loc[df['Gender'] == 'Male','Q33'].dropna(),df.loc[df['Gender'] == 'Female','Q33'].dropna(),alternative='two-sided')
p_value = '$<$0.001' if p_value < 0.001 else f'{p_value:.3f}'
rslt1['p-value'] = [np.nan,np.nan,'MW-Test:',p_value,np.nan,np.nan]
rslt1[['Male','Female']] = rslt1[['Male','Female']]*100
rslt1[['Male','Female','p-value']]


In [ ]:
df.groupby('Gender')['firmsize'].value_counts(normalize=True).unstack(0)

In [ ]:
def compare_firm_size(df):
    # Split data
    df_female = df.loc[df['Gender'] == 'Female','firmsize'].dropna()
    df_male = df.loc[df['Gender'] == 'Male','firmsize'].dropna()
    
    # Observations 
    nobs = np.array([
        len(df_male),
        len(df_female)])

    rslt = pd.DataFrame({'Native':[],'Immigrant':[],'p-value':[]})
    for firm_size in ['Less than 20 employees','Between 20 to 200 employees','Between 200 to 1,000 employees','Between 1,000 to 5,000 employees','Between 5,000 to 10,000 employees','More than 10,000 employees']:
        # Count obs
        count = np.array([
            df_male.value_counts().loc[firm_size],
            df_female.value_counts().loc[firm_size],
                       ])
        fraction = (count / nobs)*100
        z_stat, p_value = sm.stats.proportions_ztest(count, nobs, alternative='two-sided')

        p_value = '$<$0.001' if p_value < 0.001 else f'{p_value:.3f}'
        new_row = pd.DataFrame({'Male': f'{fraction[0]:.2f}', 'Female': f'{fraction[1]:.2f}', 'p-value': p_value},index=[firm_size])
        rslt = pd.concat([rslt,new_row])

    return rslt

rslt1 = compare_firm_size(df)
rslt1

### Language in the first job


In [ ]:
df.groupby('Gender')[['Q32r2','Q32r1']].value_counts(normalize=True).unstack(0)

In [ ]:
def compare_language_use(df):
    # Split data
    #df_img = df.loc[df['img'] == 1,'Q32r1'].dropna()
    #df_nat = df.loc[df['img'] == 0,'Q32r1'].dropna()
    #
    ## Observations 
    #nobs = np.array([
    #    len(df_nat),
    #    len(df_img)])

    rslt = pd.DataFrame({'Male':[],'Female':[],'p-value':[]})

    ## Count obs
    #count = np.array([
    #    df_nat.value_counts().loc[1],
    #    df_img.value_counts().loc[1],
    #               ])
    #fraction = (count / nobs)*100
    #z_stat, p_value = sm.stats.proportions_ztest(count, nobs, alternative='two-sided')
    #p_value = '$<$0.001' if p_value < 0.001 else f'{p_value:.3f}'
    #new_row1 = pd.DataFrame({'Native': f'{fraction[0]:.2f}', 'Immigrant': f'{fraction[1]:.2f}', 'p-value': p_value},index=['Use German at work'])
    #rslt = pd.concat([rslt,new_row1])
    # Split data
    df_female = df.loc[df['Gender'] == 'Female','Q32r2'].dropna()
    df_male = df.loc[df['Gender'] == 'Male','Q32r2'].dropna()
    
    # Observations 
    nobs = np.array([
        len(df_male),
        len(df_female)])

    # Count obs
    count = np.array([
        df_male.value_counts().loc[1],
        df_female.value_counts().loc[1],
                   ])
    fraction = (count / nobs)*100
    z_stat, p_value = sm.stats.proportions_ztest(count, nobs, alternative='two-sided')
    p_value = '$<$0.001' if p_value < 0.001 else f'{p_value:.3f}'
    new_row2 = pd.DataFrame({'Male': f'{fraction[0]:.2f}', 'Female': f'{fraction[1]:.2f}', 'p-value': p_value},index=['Use English at work'])
    rslt = pd.concat([rslt,new_row2])

    return rslt

rslt2 = compare_language_use(df)
rslt2

In [ ]:
rslt3 = (df.groupby(['Gender'])[['job_requirement_college_degree']].value_counts(normalize=True).unstack(0).loc[0])*100
#rslt3[['Natives','Immigrants']] = rslt3[[0,1]]
df_female = df.loc[df['Gender'] == 'Female','job_requirement_college_degree'].dropna()
df_male = df.loc[df['Gender'] == 'Male','job_requirement_college_degree'].dropna()

# Observations 
nobs = np.array([
    len(df_male),
    len(df_female)])
# Count obs
count = np.array([
    df_male.value_counts().loc[1],
    df_female.value_counts().loc[1],
               ])
fraction = (count / nobs)*100
z_stat, p_value = sm.stats.proportions_ztest(count, nobs, alternative='two-sided')
p_value = '$<$0.001' if p_value < 0.001 else f'{p_value:.3f}'
rslt3['p-value'] = p_value

rslt3 = pd.DataFrame(rslt3).T
rslt3.rename(index={0:'Job requires university degree'},inplace=True)
rslt3

In [ ]:
df['sector_new'] = df['sector_coarse'].replace({'Not Specified': 'Other'})
df['sector_new'].value_counts(normalize=True)

In [ ]:
df.groupby('Gender')['sector_coarse'].value_counts(normalize=True).unstack(0)
df['sector_new'] = df['sector_coarse'].replace({'Not Specified': 'Other'})
def compare_sector(df):
    # Split data
    df_female = df.loc[df['Gender'] == 'Female','sector_new'].dropna()
    df_male = df.loc[df['Gender'] == 'Male','sector_new'].dropna()
    
    # Observations 
    nobs = np.array([
        len(df_male),
        len(df_female)])

    rslt = pd.DataFrame({'Male':[],'Female':[],'p-value':[]})
    for sector in ['Technology and Software Development','Engineering','Consulting', 'Marketing and Advertising', 'Finance and Banking','Healthcare and Pharmaceuticals','Science','Other']:
        # Count obs
        count = np.array([
            df_male.value_counts().loc[sector],
            df_female.value_counts().loc[sector],
                       ])
        fraction = (count / nobs)*100
        z_stat, p_value = sm.stats.proportions_ztest(count, nobs, alternative='two-sided')

        p_value = '$<$0.001' if p_value < 0.001 else f'{p_value:.3f}'
        new_row = pd.DataFrame({'Male': f'{fraction[0]:.2f}', 'Female': f'{fraction[1]:.2f}', 'p-value': p_value},index=[sector])
        rslt = pd.concat([rslt,new_row])

    return rslt

rslt4 = compare_sector(df)
rslt4

In [ ]:
def generate_latex_table(rslt1,rslt2,rslt3,rslt4, filename="js_table.tex"):
    """
    Generate a LaTeX table from a DataFrame with numerical values rounded
    to two decimal places and p-values to three decimal places.
    
    Parameters:
    df (pd.DataFrame): The data to format.
    filename (str): The output .tex file name.
    """
    # LaTeX Table Header
    latex_code = r"""
        \begin{tabular}{lccc}
            \toprule
             & Male &  Female & p-value \\
            \midrule
    """
    # Add rows dynamically
    for locs, row in rslt2.iterrows():
        latex_code += f"        \\textbf{{{locs}}} & {float(row['Male']):.2f} & {float(row['Female']):.2f} & {row['p-value']} \\\ \n"
    for locs, row in rslt3.iterrows():
        latex_code += f"        \\textbf{{{locs}}} & {float(row['Male']):.2f}\% & {float(row['Female']):.2f}\% & {row['p-value']} \\\ \n"
    latex_code += f"  \\textbf{{Firm Size}} &  & &  \\\ \n"
    for locs, row in rslt1.iterrows():
        if pd.isna(row['p-value']):
            latex_code += f"\\phantom{{00}}        {locs} & {float(row['Male']):.2f}\% & {float(row['Female']):.2f}\% &  \\\ \n"
        else:
            latex_code += f"\\phantom{{00}}        {locs} & {float(row['Male']):.2f}\% & {float(row['Female']):.2f}\% & {row['p-value']} \\\ \n"
    
    latex_code += f"  \\textbf{{Sectors}} &  & &  \\\ \n"
    for locs, row in rslt4.iterrows():
        latex_code += f"\\phantom{{00}}        {locs} & {row['Male']}\% & {row['Female']}\% & {row['p-value']} \\\ \n"
    # LaTeX Table Footer
    latex_code += r"""        \bottomrule
    \end{tabular}
    """
    
    # Save to .tex file
    with open(filename, "w") as f:
        f.write(latex_code)
    
    return filename
dir = new_dir + "\\img_labormarket\\analysis\\Gender analysis\\Charts and tables\\job_characteristics.tex"
generate_latex_table(rslt1,rslt2,rslt3,rslt4, filename=dir)

In [ ]:
df['Q37'].value_counts()

In [ ]:
rslt = male_female_comparison(df, ['Q37'])
rslt

In [ ]:
df = restrict_wage(df, 'first_wage', 15, 150)
df['immediatejob_diff'] = df['Q47'] - df['Q47_1']
wages = df.groupby('Gender')['Q36r1'].mean()

wages_avg_female = wages[1]
wages_avg_male = wages[0]
df['perceived_gap'] = df['Q48r1'] - df['Q48_1r1']
df['male_deviation'] = df['Q48r1'] - wages_avg_male
df['female_deviation'] = df['Q48_1r1'] - wages_avg_female
vars = ['Q44r1','Q45', 'OutsideOption', 'first_wage' ,'male_deviation','female_deviation','Q37','Q43r1']  # Add the variables you want to test


rslt = male_female_comparison(df, vars)

rename_dict = {
    'Q44r1': 'Earnings in five years (in 1000€)',
    'Q45':'Probability to be a manager in five years',
    'Q37': 'Negotiation',
    #'Q45_a': "Imagine that you were forced to leave your current job and that you had 3 months to find a job at another employer in the same occupation. Do you think that you would find a job that would offer you a higher overall pay(1), the same pay(2) or a lower pay(3)",
    'OutsideOption': 'Outside Option',
    #'Q47': 'What percent of German graduates (on a 0-100 scale) who graduated from [pipe: Q6] do you think were working full-time immediately after graduation?',
    #'Q47_1': 'And what percent of INTERNATIONAL (non-German) graduates do you think were working full-time immediately after graduation?',
    'first_wage':'Actual Starting Wage'
    #'Q48r1': 'Belief Wage Germans',
    #'Q48_1r1': 'Belief Wage Immigrants',
    #'native_deviation':'Deviation from Native Average',
    #'male_deviation':'Deviation from Immigrant Average',
}
rslt.rename(index=rename_dict, inplace=True)
rslt[['Female Avg', 'Male Avg', 'p-value']]
rslt['p-value'] = rslt['p-value'].apply(lambda x: '$<$0.001' if float(x) < 0.001 else f'{x:.3f}')
rslt.rename(columns={'Male Avg': 'Male', 'Female Avg':'Female'},inplace=True)
rslt


In [ ]:
def compare_Negotiations(df):
    rslt = pd.DataFrame({'Male':[],'Female':[],'p-value':[]})
    df_female = df.loc[(df['Gender'] == 'Female') , 'Q37'].dropna()
    df_male = df.loc[(df['Gender'] == 'Male') , 'Q37'].dropna()

    count = np.array([df_male.value_counts().loc[1],df_female.value_counts().loc[1]])
    nobs = np.array([len(df_male),len(df_female)])
    z_stat, p_value = sm.stats.proportions_ztest(count, nobs,alternative='two-sided')

    fraction = (count / nobs)*100
    p_value = '$<$0.001' if float(p_value) < 0.001 else f'{p_value:.3f}'
    new_row1 = pd.DataFrame({'Male': f'{fraction[0]:.2f}', 'Female': f'{fraction[1]:.2f}', 'p-value': p_value},index=['Negotiated'])
    rslt = pd.concat([rslt,new_row1])
    return rslt

rslt_temp = compare_Negotiations(df)
rslt_temp


In [ ]:
insert_index = rslt.index.get_loc("Actual Starting Wage")

rslt = pd.concat([rslt.iloc[:insert_index], rslt_temp, rslt.iloc[insert_index:]])
rslt

In [ ]:
def generate_latex_table(rslt, filename="js_table.tex"):
    """
    Generate a LaTeX table from a DataFrame with numerical values rounded
    to two decimal places and p-values to three decimal places.
    
    Parameters:
    df (pd.DataFrame): The data to format.
    filename (str): The output .tex file name.
    """
    # LaTeX Table Header
    latex_code = r"""
        \begin{tabular}{lccc}
            \toprule
             & Male &  Female & p-value \\
            \midrule
    """
    # Add rows dynamically
    for locs, row in rslt.iterrows():
        if locs == 'Probability to be a manager in five years':
            latex_code += f"        {locs} & {float(row['Male']):.2f}\% & {float(row['Female']):.2f}\% & {row['p-value']} \\\ \n"
        elif locs == 'Outside Option':
            latex_code += f"        {locs} & {int(row['Male'])} & {int(row['Female'])} & {row['p-value']} \\\ \n"
        #elif locs == 'Yes to Negotiations':
         #   latex_code += f"        {locs} & {int(row['Male'])} & {int(row['Female'])} & {row['p-value']} \\\ \n"
        else:
            latex_code += f"        {locs} & {float(row['Male']):.2f} & {float(row['Female']):.2f} & {row['p-value']} \\\ \n"
    # LaTeX Table Footer
    latex_code += r"""        \bottomrule
    \end{tabular}
    """
    
    # Save to .tex file
    with open(filename, "w") as f:
        f.write(latex_code)
    
    return filename

dir = new_dir + "\\img_labormarket\\analysis\\Gender analysis\\Charts and tables\\beliefs.tex"
generate_latex_table(rslt, filename=dir)

In [ ]:
df.groupby('Gender')['OutsideOption'].mean()

In [ ]:
# Neue Kodierung: 1 = Kein Universitätsabschluss erforderlich, 2 = Universitätsabschluss erforderlich, 3 = Ich weiß es nicht
def recode_q39(value):
    if value in [1, 2, 3]:
        return 0
    elif value in [4, 5]:
        return 1
    elif value == 6:
        return np.nan
    else:
        return np.nan

df['Q39_new'] = df['Q39'].apply(recode_q39)
df.groupby(['Gender'])[['Q39_new']].value_counts()

In [ ]:
df.groupby(['Gender','Q39_new'])["Q36r1"].mean().unstack(0)

In [ ]:
df = add_codings(df, mapping_df, 'Q32r2')
df.groupby(['Gender'])['Q32r2_label'].value_counts(normalize=True).unstack(0)

In [ ]:
df.groupby('Gender')['sector_coarse'].value_counts(normalize=True).unstack(0)

In [ ]:
rslt = male_female_comparison(df, ['job_search_months','Q21','DiffGradJobStart','GradJobMonth','StartJobMonth'])
rslt

In [ ]:
df_w = restrict_wage(df, 'reservation_wage', 15, 150)
rslt4 = male_female_comparison(df_w, ['reservation_wage'])
rslt4.rename(columns={'Male Avg': 'Male', 'Female Avg': 'Female'},inplace=True)
rslt4.rename(index={'reservation_wage':'Initial Reservation Wage'},inplace=True)
rslt4 = rslt4[['Male','Female','p-value']]
rslt4

In [ ]:
rslt5 = male_female_comparison(df_w, ['job_search_months'])
rslt5.rename(columns={'Male Avg': 'Male', 'Female Avg': 'Female'},inplace=True)
rslt5.rename(index={'job_search_months':'Job Search (in months)'},inplace=True)
rslt5 = rslt5[['Male','Female','p-value']]
rslt5

5.5 months compared to almost 4 months for natives. Significant on 5%-level.

In [ ]:
mapping_df = pd.read_excel('data/ValueCodings.xlsx')
mapping_df['Frage'].fillna(method='ffill',inplace=True)
df = add_codings(df,mapping_df,'Q23','no_internships')
def compare_offer_acceptance(df):
    rslt = pd.DataFrame({'Male':[],'Female':[],'p-value':[]})
    df_img = df.loc[df['Gender'] == 'Female','Q23'].dropna()
    df_nat = df.loc[df['Gender'] == 'Male','Q23'].dropna()

    count = np.array([df_nat.value_counts().loc[1],df_img.value_counts().loc[1]])
    nobs = np.array([len(df_nat),len(df_img)])
    z_stat, p_value = sm.stats.proportions_ztest(count, nobs,alternative='two-sided')

    fraction = (count / nobs)*100
    p_value = '$<$0.001' if float(p_value) < 0.001 else f'{p_value:.3f}'
    new_row1 = pd.DataFrame({'Male': f'{fraction[0]:.2f}', 'Female': f'{fraction[1]:.2f}', 'p-value': p_value},index=['Accept first offer'])
    rslt = pd.concat([rslt,new_row1])
    return rslt

rslt6 = compare_offer_acceptance(df)
rslt6

## How many applications did you send?

In [ ]:
res = df.groupby('Gender')['no_applications'].value_counts(normalize=True).unstack()
res[['Less than 5 applications','Between 5 and 10 applications','Between 10 and 20 applications','Between 20 and 50 applications','More than 50 applications']]

In [ ]:
group1 = df[df['Gender'] == 'Female']['Q22'].dropna()
group2 = df[df['Gender'] == 'Male']['Q22'].dropna()
# Perform Mann-Whitney U test
stat, p_value = stats.mannwhitneyu(group1, group2, alternative='two-sided')

print(f"Mann-Whitney U Test: U={stat}, p={p_value}")

## When did you start searching for a job?

In [ ]:
# Create a 2x2 table
table = pd.crosstab(df['Gender'], df['Q21'].replace({1: '0-6 months before graduation', 2: 'After graduation'}))
n = table.sum(axis=1)
table = table.div(n, axis=0)*100
table

In [ ]:
# Perform Fisher’s Exact Test
odds_ratio, p_value = stats.fisher_exact(table)

print(f"Fisher’s Exact Test: OR={odds_ratio}, p={p_value}")

Immigrants start searching slightly earlier.

5 and half months to start the job compared to 2 months for natives. 

## Search Radius

In [ ]:
mapping_df = pd.read_excel('data/ValueCodings.xlsx')
mapping_df['Frage'].fillna(method='ffill',inplace=True)
df = add_codings(df, mapping_df, 'Q20','search_distance')
res = df.groupby('Gender')['search_distance'].value_counts(normalize=True).unstack()
res[['< 20 km','< 50 km', '< 100 km','within Germany','within the EU','all over the world']]

In [ ]:
group1 = df[df['Gender'] == 'Female']['Q20'].dropna().astype(int)
group2 = df[df['Gender'] == 'Male']['Q20'].dropna().astype(int)
# Perform Mann-Whitney U test
stat, p_value = stats.mannwhitneyu(group1, group2, alternative='two-sided')

print(f"Mann-Whitney U Test: U={stat}, p={p_value}")

They are likelier to only apply close and especially they are applying more boradly in particular within Germany. 

## Initial Reservation Wage

In [ ]:
df_w = restrict_wage(df, 'reservation_wage')
rslt = male_female_comparison(df_w, ['reservation_wage'])
rslt[['Female Avg', 'Male Avg', 'p-value']]

Reservation wages are very similar.

## Accept first job

In [ ]:
# Create a 2x2 table
table = pd.crosstab(df['img'], df['Q23'].replace({1: 'Yes', 2: 'No'}))
n = table.sum(axis=1)
table = table.div(n, axis=0)*100
table

In [ ]:
# Perform Fisher’s Exact Test
odds_ratio, p_value = stats.fisher_exact(table)

print(f"Fisher’s Exact Test: OR={odds_ratio}, p={p_value}")

Immigrants are likelier to accept the first job. 

# Internships

## Number of Internships?

In [ ]:
mapping_df = pd.read_excel('data/ValueCodings.xlsx')
mapping_df['Frage'].fillna(method='ffill',inplace=True)
df = add_codings(df,mapping_df,'Q16','internships')
df.groupby('Gender')[['internships']].value_counts(normalize=True).unstack(0)

In [ ]:
# Assuming `df` has a column "Group" (e.g., "Native" vs. "Immigrant") and "Applications_Sent" (coded 1–5)
group1 = df[df['Gender'] == 'Female']['Q16'].dropna().astype(int)
group2 = df[df['Gender'] == 'Male']['Q16'].dropna().astype(int)

# Perform Mann-Whitney U test
stat, p_value = stats.mannwhitneyu(group1, group2, alternative='two-sided')

print(f"Mann-Whitney U Test: U={stat}, p={p_value}")

Immigrants are having fewer internships. 

## Are you employed in the company that you worked in as an intern?

In [ ]:
# Create a 2x2 table
table = pd.crosstab(df['Gender'], df['Q17'].replace({1: 'Yes', 2: 'No'}))
n = table.sum(axis=1)
table = table.div(n, axis=0)*100
table

In [ ]:
from scipy.stats import fisher_exact

# Perform Fisher’s Exact Test
odds_ratio, p_value = fisher_exact(table)

print(f"Fisher’s Exact Test: OR={odds_ratio}, p={p_value}")

No significant or large difference

## Did you already have an offer before graduation?

TODO: I think we worded it differently

In [ ]:
# Create a 2x2 table
table = pd.crosstab(df['Gender'], df['Q18'].replace({1: 'I searched for my job', 2: 'No, I already had an offer before graduation'}))
n = table.sum(axis=1)
table = table.div(n, axis=0)*100
table

In [ ]:
# Perform Fisher’s Exact Test
odds_ratio, p_value = fisher_exact(table)

print(f"Fisher’s Exact Test: OR={odds_ratio}, p={p_value}")

There is a gap but not significant on any classic level.

# Institutional Knowledge

### What factor(s) do you believe were most influential in determining your job search outcomes?

In [ ]:
df, devs27 = likert_comparison(df, 'Q27r',q_range=9)

rslt = male_female_comparison(df,devs27)
renamed_dict = {
'Q27r1_devs':'Technical qualifications',
'Q27r2_devs':'Communication skills',
'Q27r3_devs':'University grades',
'Q27r4_devs':'Network in the industry',
'Q27r5_devs':'Internships and student jobs',
'Q27r6_devs':'Letter of recommendation',
'Q27r7_devs':'Engagement in voluntary and social activities',
'Q27r8_devs':'Gender',
'Q27r9_devs':'Nationality/ethnicity',
}
rslt = rslt.rename(index=renamed_dict)
rslt[['Female Avg', 'Male Avg', 'p-value']]
rslt['Female'] = rslt['Female Avg']
rslt['Male'] = rslt['Male Avg']
rslt[['Female', 'Male', 'p-value']]

So compared to natives immigrants value:

- University grades less
- Network in the industry matters more (Whereas natives deemed it to be a lesser point they deem it to be more important)
- LoR is deemed less important for natives
- Engagement in voluntary and social activities less
- Both belief Gender does not matter (though it does)
- They think ethnicity matters more than natives.

In [ ]:
df, devs27 = likert_comparison(df, 'Q27r',q_range=9)

rslt = male_female_comparison(df,devs27)
renamed_dict = {
'Q27r1_devs':'Technical qualifications',
'Q27r2_devs':'Communication skills',
'Q27r3_devs':'University grades',
'Q27r4_devs':'Network in the industry',
'Q27r5_devs':'Internships and student jobs',
'Q27r6_devs':'Letter of recommendation',
'Q27r7_devs':'Engagement in voluntary and social activities',
'Q27r8_devs':'Gender',
'Q27r9_devs':'Nationality/ethnicity',
}
rslt = rslt.rename(index=renamed_dict)
rslt[['Female Avg', 'Male Avg', 'p-value']]
rslt['Female'] = rslt['Female Avg']
rslt['Male'] = rslt['Male Avg']
rslt[['Female', 'Male', 'p-value']]

rslt.sort_values(by='Female', inplace=True, ascending=False)

fig, ax = plt.subplots(figsize=(16, 10))

questions = rslt.index  # Assuming rslt is the results DataFrame
ax.axvline(0, color='gray', linestyle='--', alpha=0.6, linewidth=2)  # Reference line at zero
# Plot lines connecting all native points
ax.plot(rslt["Male"], range(len(questions)), 'b-', alpha=0.9,color=green, linewidth=5, label="Male", zorder=2)
ax.scatter(rslt["Male"], range(len(questions)), color=green, s=100, linewidth=3.5, zorder=3)

# Plot lines connecting all immigrant points
ax.plot(rslt["Female"], range(len(questions)), 'r-', alpha=0.9,color=red, linewidth=5, label="Female", zorder=2)
ax.scatter(rslt["Female"], range(len(questions)), color=red, s=100, linewidth=3.5, zorder=3)

# Add p-values to the right of each question
for i, p_value in enumerate(rslt["p-value"]):
    p_text = f"p = {p_value:.3f}" if p_value >= 0.001 else "p < 0.001"
    ax.text(rslt["Male"].max() + 0.1, i, p_text, va='center', fontsize=18)

# Formatting axes and labels
ax.set_yticks(range(len(questions)))
ax.set_yticklabels(questions, fontsize=22)
ax.set_xlim(-.85,1.5)
ax.set_xlabel("Demeaned Coefficient", fontsize=22)
ax.set_xticks(np.arange(-.8, 1.2, 0.2))
ax.set_xticklabels([f"{tick:.1f}" for tick in ax.get_xticks()], fontsize=18)
ax.tick_params(axis='x', labelsize=18)
ax.legend(fontsize=18, loc='lower left')
#ax.grid(linestyle='--', alpha=0.6)
#plt.savefig(BLD / 'figures' / 'job_search_factors.pdf', bbox_inches='tight')
# Show the updated plot
plt.show()


### What would you change if you were to search for a first job after college all over again?

In [ ]:
df, devs29 = likert_comparison(df, 'Q29r',q_range=7)

rslt = male_female_comparison(df,devs29)

renamed_dict = {
'Q29r1_devs': 'Q29r1: Use different search channels',
'Q29r2_devs': 'Q29r2: Do more internships',
'Q29r3_devs': 'Q29r3: Use a more appropriate and professional CV',
'Q29r4_devs': 'Q29r4: Negotiate salary',
'Q29r5_devs': 'Q29r5: Tailor each application to the specific job',
'Q29r6_devs': 'Q29r6: Prepare more for job interviews',
'Q29r7_devs': 'Q29r7: Obtain more information about the types of jobs and salary ranges available for my qualifications',
}

rslt = male_female_comparison(df, devs29)
rslt = rslt.rename(index=renamed_dict)
rslt[['Female Avg', 'Male Avg', 'p-value']]

Does not look terribly interesting across native and immigrants. Yet, it seems obtaining information (Q29r7) and negotiating (Q29r4) are the biggest items mentioned. 